# History Recorder Playground
Use this notebook to tweak manual scenarios, capture history snapshots, and preview quick plots.

## How to use this notebook
1. Adjust the station definitions or scenario rows below.
2. Re-run the helper cell to regenerate the `HistoryRecorder`.
3. Pull the resulting series into your plotting library of choice.
4. (Optional) Run the final cell to generate a Matplotlib preview.

In [ ]:
from stations import StationProfile
from bikes import derive_inventory
from history import HistoryRecorder

station_profiles = [
    StationProfile(identifier="hub-1", name="Transit Hub", capacity=24),
    StationProfile(identifier="park-1", name="Riverside Park", capacity=18),
]
profile_by_id = {profile.identifier: profile for profile in station_profiles}

def run_manual_scenario(scenario_rows):
    """Return a HistoryRecorder populated from the provided rows."""
    recorder = HistoryRecorder()
    for row in scenario_rows:
        inventories = [
            derive_inventory(profile_by_id[station_id], row["docked_bikes"][station_id])
            for station_id in profile_by_id
        ]
        recorder.record(minute=row["minute"], rider_states=row["states"], station_snapshots=inventories)
    return recorder

In [ ]:
scenario_rows = [
    {
        "minute": 0,
        "states": {"waiting_for_bike": 12, "riding": 0, "returning": 0},
        "docked_bikes": {"hub-1": 20, "park-1": 14},
    },
    {
        "minute": 30,
        "states": {"waiting_for_bike": 8, "riding": 6, "returning": 1},
        "docked_bikes": {"hub-1": 16, "park-1": 11},
    },
    {
        "minute": 60,
        "states": {"waiting_for_bike": 4, "riding": 8, "returning": 3},
        "docked_bikes": {"hub-1": 13, "park-1": 9},
    },
    {
        "minute": 90,
        "states": {"waiting_for_bike": 6, "riding": 5, "returning": 5},
        "docked_bikes": {"hub-1": 15, "park-1": 10},
    },
]

recorder = run_manual_scenario(scenario_rows)
print(f"Recorded {len(recorder.snapshots())} snapshots.")

In [ ]:
waiting_series = recorder.rider_state_series("waiting_for_bike")
hub_series = recorder.station_bike_series("hub-1")
waiting_series, hub_series

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Install matplotlib via `%pip install matplotlib` to enable plotting.")
else:
    minutes, waiting_counts = zip(*waiting_series)
    _, hub_counts = zip(*hub_series)
    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(minutes, waiting_counts, label="Waiting riders", color="tab:blue")
    ax1.set_xlabel("Minute")
    ax1.set_ylabel("Riders", color="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot(minutes, hub_counts, label="Hub bikes", color="tab:orange")
    ax2.set_ylabel("Hub bikes", color="tab:orange")
    plt.title("Sample rider vs. bike availability trend")
    fig.tight_layout()
    plt.show()